# F01 Transformer Shapes and Attention Reading


## What this foundation lab is for

This lab exists to reduce the most common beginner failure modes before the article-first path starts.


## Skills you should leave with

- Explain the relationship between tokens, embeddings, attention scores, and context vectors.
- Read an attention heatmap directly instead of saying only 'the model looks here'.
- Treat the residual stream as an accumulating computation cache.


## Ship these outputs

- One hand-annotated attention heatmap.
- One residual-update explanation written in plain language.
- One notebook annotation that makes the tensor shapes explicit.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/circuits-zoom-in-fresh-20260325.git"
REPO_DIR = "circuits-zoom-in-fresh-20260325"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import math
import matplotlib.pyplot as plt
import numpy as np

tokens = ["Ada", "wrote", "the", "patch", "carefully"]
embeddings = np.array([
    [1.0, 0.1, 0.3],
    [0.8, 0.4, 0.2],
    [0.2, 0.9, 0.1],
    [0.7, 0.3, 0.9],
    [0.9, 0.2, 0.8],
])
Wq = np.array([[1.0, 0.2], [0.1, 0.9], [0.7, 0.3]])
Wk = np.array([[0.9, 0.1], [0.2, 1.0], [0.6, 0.4]])
Wv = np.array([[0.4, 0.8], [0.9, 0.3], [0.3, 0.7]])

Q = embeddings @ Wq
K = embeddings @ Wk
V = embeddings @ Wv
scores = Q @ K.T / math.sqrt(K.shape[-1])
scores = scores - scores.max(axis=-1, keepdims=True)
weights = np.exp(scores)
weights = weights / weights.sum(axis=-1, keepdims=True)
contexts = weights @ V
residual_after = np.concatenate([embeddings[:, :2] + contexts, embeddings[:, 2:]], axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(weights, cmap="Blues")
axes[0].set_xticks(range(len(tokens)), tokens, rotation=30)
axes[0].set_yticks(range(len(tokens)), tokens)
axes[0].set_title("Attention weights")
plt.colorbar(im, ax=axes[0], fraction=0.046)

axes[1].plot(residual_after[-1], marker="o", color="#c96a28")
axes[1].set_title("Final-token residual update")
axes[1].set_xlabel("channel")
plt.tight_layout()

print("Q shape:", Q.shape, "| K shape:", K.shape, "| V shape:", V.shape)
print("Most attended token for the final position:", tokens[int(weights[-1].argmax())])
print("Final-position context vector:", np.round(contexts[-1], 3))


## Takeaway

Before you can read circuits, you need to read shapes, attention matrices, and residual updates without hand-waving.


## Self-check questions

- Why does high attention weight not automatically mean high causal contribution?
- Why is the residual stream a better language for later tracing than 'one single neuron'?
- If you do not understand the shapes of Q, K, and V, where do later circuit explanations break down?
- If you cannot answer at least two questions without reopening the notebook, stay here before moving to the article track.
